In [1]:
#this notebook is used to create the baseline calculations for each "followed" token (configured in the config file), using spark
#setup installing package to the venv
%pip install pandas pyarrow pyspark
%pip install install-jdk

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
#java home setup
#this is importent for the venv, not part of our project
%pip install install-jdk

import os
import sys
import glob
import jdk

# 1. Target the .venv directory of the current running Python kernel
venv_path = sys.prefix
jvm_dir = os.path.join(venv_path, "jvm")

# 2. Download/ensure JDK 17 exists for THIS current OS/architecture
jdk.install('17', path=jvm_dir)

# 3. Dynamically search for the 'bin/java' executable regardless of OS directory nesting
java_execs = glob.glob(os.path.join(jvm_dir, "**/bin/java"), recursive=True)
if not java_execs:
    # Check for Windows .exe just in case
    java_execs = glob.glob(os.path.join(jvm_dir, "**/bin/java.exe"), recursive=True)

if not java_execs:
    raise RuntimeError(f"JDK binary could not be found inside {jvm_dir}")

# The true JAVA_HOME is the parent directory of 'bin'
resolved_java_home = os.path.dirname(os.path.dirname(os.path.abspath(java_execs[0])))

# 4. Set environment variables for the active session
os.environ["JAVA_HOME"] = resolved_java_home
os.environ["PATH"] = os.path.join(resolved_java_home, "bin") + os.pathsep + os.environ.get("PATH", "")

print(f"Universal JDK configured at: {resolved_java_home}")

Note: you may need to restart the kernel to use updated packages.
Universal JDK configured at: /home/wnder/Documents/repos/teleSpikeRedo/.venv/jvm/jdk-17.0.20.1+1


In [3]:
import re
import json
import sqlite3
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as SqlFun
from pyspark.sql.types import ArrayType, StringType

with open("config.json", "r") as f:
    config = json.load(f)
#def GetBaseline(token):

# a simple worker functaion for parsing one message into tokes
def tokenize(text):
    if not text:
        return []
    #Hebrew and alphanumeric words of length >= 2
    return re.findall(r"[\u0590-\u05fe\w]{2,}", text.lower())

#conver time stamp into hour and date
def add_time_columns(df):
    return df.withColumn("dt", SqlFun.to_timestamp(SqlFun.col("ts"))) \
             .withColumn("hour", SqlFun.hour(SqlFun.col("dt"))) \
             .withColumn("date", SqlFun.to_date(SqlFun.col("dt")))

#tokenise and exploead 
def explode_and_filter_tokens(df, allowed_tokens=None):
    tokenize_udf = SqlFun.udf(tokenize, ArrayType(StringType()))
    
    tokens_df = df.withColumn("word", SqlFun.explode(tokenize_udf(SqlFun.col("text"))))
    
    if allowed_tokens:
        tokens_df = tokens_df.filter(SqlFun.col("word").isin(allowed_tokens))
        
    return tokens_df

def compute_hourly_pivots(tokens_df, total_days):
    # group rows per word and hour
    counts = tokens_df.groupBy("word", "hour").count()
    
    # Pivot hours into columns (now each word has a clolumb for each time of day)
    pivoted = counts.groupBy("word").pivot("hour", list(range(24))).sum("count").na.fill(0)
    
    # divide counts by total days to get average baseline per hour
    # Rename columns to h0, h1 ... h23
    for h in range(24):
        pivoted = pivoted.withColumn(f"h{h}", SqlFun.col(str(h)) / total_days).drop(str(h))
        
    return pivoted

#main function calling other parts    
def generate_baseline_table(sqlite_path, output_table_path, allowed_tokens):
    #this will recive an sql full of thounsds or millions of messages and a 
    #list of importent tokens. it will clean and tokenise each message, filter not importent tokens like "and" "if". any thing that isnt in the list.
    #it will then save in a small sql table
    #a row for each token with a columb for each hour of the day, and save the avrage apperenses in that hour for each token. we can than devide by 60 or 240 to get baselines for our time window

    spark = SparkSession.builder \
        .appName("BaselineGenerator") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()

    # load messages from local data base into Spark
    conn = sqlite3.connect(sqlite_path)
    pdf = pd.read_sql_query("SELECT text, ts FROM messages", conn)
    conn.close()

    if pdf.empty:
        print("No messages found.")
        return

    raw_df = spark.createDataFrame(pdf)

    #add time of day columb to data frame, parsed from the timestamp that came with the message
    #as well as a date columb for later calcualtions
    timed_df = add_time_columns(raw_df)

    # Calculate total days for avrge calculations later on
    total_days = max(1, timed_df.select("date").distinct().count())

    # tokenization and explode into rows
    tokens_df = explode_and_filter_tokens(timed_df, allowed_tokens=allowed_tokens)

    # use the data to turn the table into each row has: word h0 h1...., in each cloumb have the avrage for that time of day
    baseline_matrix = compute_hourly_pivots(tokens_df, total_days)

    # save to sql
    baseline_pdf = baseline_matrix.toPandas()
    
    conn_out = sqlite3.connect(output_table_path)
    baseline_pdf.to_sql("token_baselines", conn_out, if_exists="replace", index=False)
    conn_out.close()
    
    print(f"Generated baselines for {len(baseline_pdf)} tokens over {total_days} days.")
    return baseline_pdf

In [4]:
#create baselines.db
conn = sqlite3.connect(config["baselines_db_path"])
cursor = conn.cursor()

hour_cols = ", ".join([f"h{i} REAL" for i in range(24)])

cursor.execute(f"""
CREATE TABLE IF NOT EXISTS token_baselines (
    word TEXT,
    {hour_cols}
)
""")

conn.commit()
conn.close()

print("Empty token_baselines table created.")

Empty token_baselines table created.


In [5]:

# Run the pipeline on your scraped messages
baselines_df = generate_baseline_table(
    sqlite_path=config["messages_db_path"],#"messages.db",
    output_table_path=config["baselines_db_path"],
    allowed_tokens=config["followed_tokens"]  # Set to a list like ["טיל", "אזעקה"] if you want to filter, or None for all
)

# Preview the top tokens at 14:00 (2:00 PM)
if baselines_df is not None:
    print(baselines_df[["word", "h14"]].sort_values(by="h14", ascending=False).head(10))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/31 10:22:54 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 10.100.102.22 instead (on interface enp0s31f6)
26/08/31 10:22:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/wnder/Documents/repos/teleSpikeRedo/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/31 10:22:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
/home/wnder/Documents/repos/tele

Generated baselines for 103 tokens over 192 days.
        word       h14
15     ישראל  0.500000
90     איראן  0.244792
73     לבנון  0.161458
40  חיזבאללה  0.156250
83     העורף  0.114583
29      מפקד  0.104167
54     פיקוד  0.104167
25    אזעקות  0.098958
5      ארגון  0.098958
41     כוחות  0.098958


In [ ]:
#print data base messages.db
conn = sqlite3.connect(config["messages_db_path"])
df_msgs = pd.read_sql_query("SELECT * FROM messages LIMIT 20", conn)
conn.close()
print("Messages Table:")
display(df_msgs)

In [6]:
#print basline.db
conn = sqlite3.connect(config["baselines_db_path"])
#df_base = pd.read_sql_query("SELECT word, h0, h8, h14, h20 FROM token_baselines ORDER BY h14 DESC LIMIT 10", conn)
df_base = pd.read_sql_query("SELECT * FROM token_baselines ORDER BY h14 DESC LIMIT 10", conn)
conn.close()
print("Baselines Table:")
display(df_base)

Baselines Table:


,word,h0,h1,h2,h3,h4,h5,h6,h7,h8,...,h14,h15,h16,h17,h18,h19,h20,h21,h22,h23
0,ישראל,0.192708,0.114583,0.177083,0.036458,0.000000,0.005208,0.015625,0.177083,0.255208,...,0.500000,0.427083,0.458333,0.583333,0.447917,0.619792,0.666667,0.526042,0.395833,0.270833
1,איראן,0.229167,0.302083,0.234375,0.026042,0.036458,0.005208,0.041667,0.135417,0.229167,...,0.244792,0.286458,0.317708,0.416667,0.270833,0.322917,0.552083,0.416667,0.265625,0.197917
2,לבנון,0.119792,0.020833,0.036458,0.020833,0.000000,0.000000,0.072917,0.130208,0.114583,...,0.161458,0.156250,0.166667,0.229167,0.239583,0.276042,0.281250,0.223958,0.218750,0.161458
3,חיזבאללה,0.093750,0.036458,0.041667,0.000000,0.000000,0.000000,0.015625,0.072917,0.088542,...,0.156250,0.119792,0.203125,0.104167,0.114583,0.276042,0.229167,0.171875,0.140625,0.109375
4,העורף,0.109375,0.093750,0.031250,0.020833,0.005208,0.005208,0.020833,0.041667,0.041667,...,0.114583,0.114583,0.067708,0.098958,0.104167,0.114583,0.171875,0.098958,0.093750,0.114583
5,מפקד,0.020833,0.010417,0.015625,0.010417,0.000000,0.000000,0.005208,0.005208,0.020833,...,0.104167,0.088542,0.020833,0.067708,0.041667,0.083333,0.151042,0.062500,0.046875,0.005208
6,פיקוד,0.145833,0.072917,0.031250,0.005208,0.005208,0.005208,0.036458,0.062500,0.046875,...,0.104167,0.093750,0.046875,0.098958,0.098958,0.109375,0.166667,0.114583,0.114583,0.098958
7,ארגון,0.052083,0.005208,0.015625,0.000000,0.000000,0.000000,0.015625,0.031250,0.041667,...,0.098958,0.078125,0.098958,0.072917,0.052083,0.052083,0.109375,0.119792,0.052083,0.046875
8,אזעקות,0.083333,0.114583,0.036458,0.020833,0.000000,0.010417,0.000000,0.046875,0.046875,...,0.098958,0.104167,0.114583,0.130208,0.088542,0.057292,0.083333,0.088542,0.078125,0.067708
9,כוחות,0.088542,0.020833,0.020833,0.010417,0.000000,0.000000,0.031250,0.046875,0.114583,...,0.098958,0.119792,0.135417,0.151042,0.114583,0.151042,0.151042,0.104167,0.078125,0.104167
